# Shared Next-POI GRU With Service Constraints

This notebook prepares one next-POI GRU for both `/recommend` and `/generate-course` and stops before the full experiment. It follows the service constraints checked from `EDA_POI.ipynb`: 5-digit area codes, custom 1-7 travel personas, capped companion count, multi-hot `preferredArea`, `travel_id`-level splits, full-data vocabulary with a BOS token, `(travel_id, day_index, area_code)` sequences, stable token artifacts, categorical ID-code handling, class-imbalance checks, and explicit missing-context tokens.


In [ ]:
from __future__ import annotations

import json
import math
import pickle
import random
import re
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import torch
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
MODEL_VERSION = "shared-next-poi-gru-v1"
VOCAB_VERSION = "full-data-content-vocab-v1"
MISSING_CONTEXT_TOKEN = "__MISSING_CONTEXT__"
PAD_TOKEN = "<PAD>"
BOS_TOKEN = "<BOS>"
SPECIAL_TOKEN_IDS = {0, 1}
COURSE_POIS_PER_DAY = 6
MIN_SEQUENCE_LEN = 1
MAX_SEQUENCE_LEN = 20
TOP_KS = (1, 3, 5, 10)

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = ROOT / "data" / "processed" / "final"
ARTIFACT_DIR = ROOT / "artifacts" / "shared_next_poi_gru_experiment" / MODEL_VERSION
TRAVELER_PATH = DATA_DIR / "final_traveler_features.csv"
TRAVEL_PATH = DATA_DIR / "final_travel_features.csv"
SEQUENCE_PATH = DATA_DIR / "final_travel_sequence.csv"

AREA_CODE_TO_NAME = {
    "11000": "서울",
    "41110": "수원",
    "28000": "인천",
    "30000": "대전",
    "27000": "대구",
    "12000": "광주",
    "26000": "부산",
    "48120": "창원",
}
AREA_NAME_TO_CODE = {name: code for code, name in AREA_CODE_TO_NAME.items()}
SIDO_TO_SERVICE_AREA_CODE = {
    "11": "11000",
    "26": "26000",
    "27": "27000",
    "28": "28000",
    "29": "12000",
    "30": "30000",
    "41": "41110",
    "48": "48120",
}
RAW_THEME_TO_PERSONA = {
    "22": 1,
    "1": 2,
    "4": 2,
    "3": 3,
    "6": 3,
    "2": 4,
    "9": 4,
    "11": 4,
    "25": 4,
    "5": 5,
    "8": 5,
    "27": 5,
    "24": 5,
    "21": 6,
    "10": 6,
    "23": 6,
    "26": 7,
    "12": 7,
    "13": 7,
}
PERSONA_LABELS = {
    1: "hotplace_sentiment",
    2: "city_shopping",
    3: "culture_history",
    4: "play_experience",
    5: "nature_activity",
    6: "rest_wellness",
    7: "contents_special",
}
AREA_CODE_PATTERN = re.compile(r"^\d{5}$")


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def normalize_context(value: Any) -> str:
    if value is None or pd.isna(value):
        return MISSING_CONTEXT_TOKEN
    text = str(value).strip()
    return text if text else MISSING_CONTEXT_TOKEN


def normalize_content_id_value(value: Any) -> str:
    text = normalize_context(value)
    if text == MISSING_CONTEXT_TOKEN:
        return ""
    return text[:-2] if text.endswith(".0") else text


def normalize_area_code_5digit(value: Any) -> str:
    text = normalize_context(value)
    if text == MISSING_CONTEXT_TOKEN:
        return MISSING_CONTEXT_TOKEN
    text = text[:-2] if text.endswith(".0") else text
    digits = "".join(ch for ch in text if ch.isdigit())
    if len(digits) == 5:
        return digits
    if len(digits) == 2:
        return SIDO_TO_SERVICE_AREA_CODE.get(digits, f"{digits}000")
    return MISSING_CONTEXT_TOKEN


def normalize_preferred_area_list(value: Any) -> list[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return [MISSING_CONTEXT_TOKEN]
    if isinstance(value, str):
        raw_values = [part for part in re.split(r"[;,\s]+", value.strip()) if part]
    elif isinstance(value, Iterable):
        raw_values = list(value)
    else:
        raw_values = [value]
    normalized = []
    for item in raw_values:
        code = normalize_area_code_5digit(item)
        if code not in normalized:
            normalized.append(code)
    return normalized or [MISSING_CONTEXT_TOKEN]


def map_raw_theme_to_persona(value: Any) -> str:
    text = normalize_context(value)
    if text == MISSING_CONTEXT_TOKEN:
        return MISSING_CONTEXT_TOKEN
    text = text[:-2] if text.endswith(".0") else text
    if text in RAW_THEME_TO_PERSONA:
        return str(RAW_THEME_TO_PERSONA[text])
    return MISSING_CONTEXT_TOKEN


def normalize_service_persona(value: Any) -> str:
    text = normalize_context(value)
    if text == MISSING_CONTEXT_TOKEN:
        return MISSING_CONTEXT_TOKEN
    text = text[:-2] if text.endswith(".0") else text
    try:
        parsed = int(text)
    except ValueError:
        return MISSING_CONTEXT_TOKEN
    return str(parsed) if 1 <= parsed <= 7 else MISSING_CONTEXT_TOKEN


def cap_companion_count(value: Any) -> str:
    try:
        parsed = int(float(str(value).strip()))
    except (TypeError, ValueError):
        return MISSING_CONTEXT_TOKEN
    return str(min(max(parsed, 0), 2))


def mode_or_first(values: pd.Series) -> str:
    clean = values.dropna().astype(str)
    if clean.empty:
        return ""
    modes = clean.mode()
    return str(modes.iloc[0] if not modes.empty else clean.iloc[0])


seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


## 1. Load CSVs And Normalize Service Context


In [ ]:
traveler_raw = pd.read_csv(TRAVELER_PATH, encoding="utf-8-sig")
travel_raw = pd.read_csv(TRAVEL_PATH, encoding="utf-8-sig")
sequence_raw = pd.read_csv(SEQUENCE_PATH, encoding="utf-8-sig")

required = {
    "traveler": {"traveler_id", "residence_code", "gender", "age_grp", "travel_style", "travel_like_code"},
    "travel": {"travel_id", "traveler_id", "companion_count", "theme"},
    "sequence": {"travel_id", "day_index", "visit_order", "visit_area_nm", "content_id", "area"},
}
assert required["traveler"].issubset(traveler_raw.columns)
assert required["travel"].issubset(travel_raw.columns)
assert required["sequence"].issubset(sequence_raw.columns)
assert all(AREA_CODE_PATTERN.fullmatch(code) for code in AREA_CODE_TO_NAME)

traveler_df = traveler_raw.copy()
traveler_df["residence_area_code"] = traveler_df["residence_code"].map(normalize_area_code_5digit)
traveler_df["preferred_area_codes"] = traveler_df["travel_like_code"].map(lambda value: normalize_preferred_area_list([value]))
traveler_df["gender"] = traveler_df["gender"].map(normalize_context)
traveler_df["age_grp"] = traveler_df["age_grp"].map(normalize_context)
traveler_df["traveler_style"] = traveler_df["travel_style"].map(normalize_context)
traveler_df = traveler_df[["traveler_id", "residence_area_code", "preferred_area_codes", "gender", "age_grp", "traveler_style"]]

travel_df = travel_raw.copy()
raw_theme_codes = travel_df["theme"].map(normalize_context).map(lambda text: text[:-2] if text.endswith(".0") else text)
travel_df["travel_persona"] = travel_df["theme"].map(map_raw_theme_to_persona)
unmapped_theme_counts = raw_theme_codes.loc[~raw_theme_codes.isin(RAW_THEME_TO_PERSONA)].value_counts(dropna=False).sort_index().to_dict()
travel_df["companion_count_bucket"] = travel_df["companion_count"].map(cap_companion_count)
travel_df = travel_df[["travel_id", "traveler_id", "companion_count_bucket", "travel_persona"]]

seq = sequence_raw.copy()
seq["content_id"] = seq["content_id"].map(normalize_content_id_value)
seq["area_name"] = seq["area"].astype(str).str.strip()
seq["content_name"] = seq["visit_area_nm"].astype(str).str.strip()
seq["area_code"] = seq["area_name"].map(AREA_NAME_TO_CODE).fillna(MISSING_CONTEXT_TOKEN)
seq["day_index"] = pd.to_numeric(seq["day_index"], errors="coerce").astype("Int64")
seq["visit_order"] = pd.to_numeric(seq["visit_order"], errors="coerce").astype("Int64")
seq = seq.loc[seq["content_id"].ne("") & seq["area_code"].ne(MISSING_CONTEXT_TOKEN)].copy()
seq = seq.sort_values(["travel_id", "day_index", "visit_order"], kind="mergesort")
assert seq["area_code"].map(lambda code: bool(AREA_CODE_PATTERN.fullmatch(str(code)))).all()

coverage = {
    "traveler_rows": len(traveler_raw),
    "travel_rows": len(travel_raw),
    "sequence_rows": len(sequence_raw),
    "supported_sequence_rows": len(seq),
    "raw_theme_unique": sorted(map(str, travel_raw["theme"].dropna().unique())),
    "service_persona_unique": sorted(travel_df["travel_persona"].unique()),
    "unmapped_raw_theme_counts": unmapped_theme_counts,
    "companion_bucket_counts": travel_df["companion_count_bucket"].value_counts(dropna=False).sort_index().to_dict(),
}
print(json.dumps(coverage, ensure_ascii=False, indent=2))
traveler_df.head()


## 2. Build `(travel_id, day_index, area_code)` Sequences And Full-Data POI Vocabulary

Sequences are grouped per travel day, not across the full trip, so the model does not learn a boundary transition from yesterday's final POI to today's first POI. The vocabulary is built before splitting from all supported POIs, because serving must use stable token IDs across train/validation/test and model versions.


In [ ]:
poi_master_base = (
    seq.groupby("content_id", as_index=False)
    .agg(
        area_code=("area_code", mode_or_first),
        area_name=("area_name", mode_or_first),
        content_name=("content_name", mode_or_first),
        visit_count=("travel_id", "size"),
    )
    .sort_values(["area_code", "content_id"], kind="mergesort")
    .reset_index(drop=True)
)

all_content_ids = sorted(poi_master_base["content_id"].astype(str).tolist())
master_content_ids = set(all_content_ids)
content_id_to_token = {PAD_TOKEN: 0, BOS_TOKEN: 1}
content_id_to_token.update({content_id: idx + 2 for idx, content_id in enumerate(all_content_ids)})
token_to_content_id = {token_id: content_id for content_id, token_id in content_id_to_token.items()}

poi_master = poi_master_base.copy()
poi_master["token_id"] = poi_master["content_id"].map(content_id_to_token).astype("Int64")
poi_master["model_version"] = MODEL_VERSION
poi_master["vocab_version"] = VOCAB_VERSION
model_poi_master = poi_master.copy()

# Deduplicate repeated POIs inside a travel day/area, following the EDA_POI pattern.
day_seq_source = (
    seq.sort_values(["travel_id", "day_index", "visit_order", "content_id"], kind="mergesort")
    .drop_duplicates(["travel_id", "day_index", "area_code", "content_id"], keep="first")
)
travel_day_area_counts = day_seq_source.groupby(["travel_id", "day_index"])["area_code"].nunique()
mixed_travel_day_count = int((travel_day_area_counts > 1).sum())
transition_source = day_seq_source.sort_values(["travel_id", "day_index", "visit_order", "content_id"], kind="mergesort")
same_travel_day_as_previous = transition_source[["travel_id", "day_index"]].eq(transition_source[["travel_id", "day_index"]].shift()).all(axis=1)
cross_area_transition_count_before_area_split = int((same_travel_day_as_previous & transition_source["area_code"].ne(transition_source["area_code"].shift())).sum())
day_sequences = (
    day_seq_source.groupby(["travel_id", "day_index", "area_code"], sort=False)
    .agg(content_sequence=("content_id", list), area_sequence=("area_code", list))
    .reset_index()
)
day_sequences["sequence_len"] = day_sequences["content_sequence"].map(len)

trip_features = travel_df.merge(traveler_df, on="traveler_id", how="inner", validate="many_to_one")
model_input = trip_features.merge(day_sequences, on="travel_id", how="inner", validate="one_to_many")
model_input = model_input.loc[model_input["sequence_len"] >= MIN_SEQUENCE_LEN].copy()
model_input["travel_day_id"] = model_input["travel_id"].astype(str) + "::" + model_input["day_index"].astype(str) + "::" + model_input["area_code"].astype(str)
model_input = model_input.reset_index(drop=True)

print("poi master rows:", len(poi_master))
print("vocab size:", len(content_id_to_token))
print("mixed travel-day rows before area split:", mixed_travel_day_count)
print("cross-area transitions before area split:", cross_area_transition_count_before_area_split)
print("model_input day rows:", len(model_input))
print(model_input["sequence_len"].describe())
model_input.head()


## 3. Split By `travel_id`, Encode Context, And Create Next-POI Samples


In [ ]:
unique_travel_ids = pd.Series(model_input["travel_id"].unique())
train_trips, temp_trips = train_test_split(unique_travel_ids, test_size=0.30, random_state=SEED, shuffle=True)
valid_trips, test_trips = train_test_split(temp_trips, test_size=0.50, random_state=SEED, shuffle=True)
split_sets = {"train": set(train_trips), "valid": set(valid_trips), "test": set(test_trips)}
assert split_sets["train"].isdisjoint(split_sets["valid"])
assert split_sets["train"].isdisjoint(split_sets["test"])
assert split_sets["valid"].isdisjoint(split_sets["test"])

train_df = model_input.loc[model_input["travel_id"].isin(split_sets["train"])].reset_index(drop=True)
valid_df = model_input.loc[model_input["travel_id"].isin(split_sets["valid"])].reset_index(drop=True)
test_df = model_input.loc[model_input["travel_id"].isin(split_sets["test"])].reset_index(drop=True)

SCALAR_CONTEXT_COLUMNS = [
    "companion_count_bucket",
    "travel_persona",
    "residence_area_code",
    "gender",
    "age_grp",
    "traveler_style",
]
PREFERRED_AREA_COLUMN = "preferred_area_codes"

class ContextFeatureEncoder:
    def __init__(self) -> None:
        self.scalar_encoder = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value=MISSING_CONTEXT_TOKEN, keep_empty_features=True)),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]
        )
        self.preferred_encoder = MultiLabelBinarizer()
        self.output_dim: int | None = None

    def _scalar_frame(self, df: pd.DataFrame) -> pd.DataFrame:
        return df[SCALAR_CONTEXT_COLUMNS].map(normalize_context).astype(str)

    def _preferred_values(self, df: pd.DataFrame) -> list[list[str]]:
        return [normalize_preferred_area_list(value) for value in df[PREFERRED_AREA_COLUMN]]

    def fit(self, df: pd.DataFrame) -> "ContextFeatureEncoder":
        scalar_features = self.scalar_encoder.fit_transform(self._scalar_frame(df))
        preferred_features = self.preferred_encoder.fit_transform(self._preferred_values(df)).astype(np.float32)
        self.output_dim = int(scalar_features.shape[1] + preferred_features.shape[1])
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        scalar_features = self.scalar_encoder.transform(self._scalar_frame(df)).astype(np.float32)
        preferred_features = self.preferred_encoder.transform(self._preferred_values(df)).astype(np.float32)
        return np.hstack([scalar_features, preferred_features]).astype(np.float32)

feature_encoder = ContextFeatureEncoder().fit(train_df)
USER_FEATURE_DIM = int(feature_encoder.output_dim or 0)

def train_input_content_ids_from_rows(df: pd.DataFrame) -> set[str]:
    content_ids: set[str] = set()
    for sequence in df["content_sequence"]:
        normalized = [normalize_content_id_value(value) for value in sequence]
        content_ids.update(content_id for content_id in normalized[:-1] if content_id in master_content_ids)
    return content_ids

def train_output_content_ids_from_rows(df: pd.DataFrame) -> set[str]:
    content_ids: set[str] = set()
    for sequence in df["content_sequence"]:
        normalized = [normalize_content_id_value(value) for value in sequence]
        content_ids.update(content_id for content_id in normalized if content_id in master_content_ids)
    return content_ids

train_known_input_content_ids = train_input_content_ids_from_rows(train_df)
train_output_candidate_content_ids = train_output_content_ids_from_rows(train_df)
unseen_by_train_sample_content_ids = master_content_ids - (train_known_input_content_ids | train_output_candidate_content_ids)
model_known_poi_metadata = {
    "model_version": MODEL_VERSION,
    "vocab_version": VOCAB_VERSION,
    "master_content_id_count": len(master_content_ids),
    "train_known_input_content_id_count": len(train_known_input_content_ids),
    "train_output_candidate_content_id_count": len(train_output_candidate_content_ids),
    "unseen_by_train_sample_content_id_count": len(unseen_by_train_sample_content_ids),
    "train_known_input_definition": "content_id appears in at least one train prefix position before a next-POI label",
    "train_output_candidate_definition": "content_id appears in at least one train label, including BOS-to-first-POI labels",
}

def project_known_sequence(content_id_sequence: Iterable[Any]) -> list[str]:
    known = []
    for value in content_id_sequence:
        content_id = normalize_content_id_value(value)
        if content_id in train_known_input_content_ids:
            known.append(content_id)
    return known

def tokens_for_known_sequence(content_id_sequence: Iterable[Any]) -> list[int]:
    return [int(content_id_to_token[content_id]) for content_id in project_known_sequence(content_id_sequence)]

def tokens_for_model_prefix(content_id_sequence: Iterable[Any]) -> list[int]:
    known_tokens = tokens_for_known_sequence(content_id_sequence)
    max_known_tokens = max(MAX_SEQUENCE_LEN - 1, 0)
    return [int(content_id_to_token[BOS_TOKEN]), *known_tokens[-max_known_tokens:]]

def make_next_poi_samples(df: pd.DataFrame) -> list[dict[str, Any]]:
    encoded = feature_encoder.transform(df)
    samples: list[dict[str, Any]] = []
    for row_idx, row in df.reset_index(drop=True).iterrows():
        content_sequence = [normalize_content_id_value(value) for value in row["content_sequence"]]
        for target_pos in range(len(content_sequence)):
            label_content_id = content_sequence[target_pos]
            prefix_tokens = tokens_for_model_prefix(content_sequence[:target_pos])
            samples.append(
                {
                    "travel_id": row["travel_id"],
                    "day_index": int(row["day_index"]),
                    "travel_day_id": row["travel_day_id"],
                    "user_features": encoded[row_idx],
                    "prefix": prefix_tokens,
                    "label": int(content_id_to_token[label_content_id]),
                    "label_content_id": label_content_id,
                    "sample_kind": "first_poi" if target_pos == 0 else "next_poi",
                }
            )
    return samples

train_samples = make_next_poi_samples(train_df)
valid_samples = make_next_poi_samples(valid_df)
test_samples = make_next_poi_samples(test_df)
split_sizes = {"train": len(train_df), "valid": len(valid_df), "test": len(test_df)}
sample_sizes = {"train": len(train_samples), "valid": len(valid_samples), "test": len(test_samples)}
sample_kind_counts = {
    split: pd.Series([sample["sample_kind"] for sample in samples]).value_counts().sort_index().to_dict()
    for split, samples in {"train": train_samples, "valid": valid_samples, "test": test_samples}.items()
}
print("split_sizes", split_sizes)
print("sample_sizes", sample_sizes)
print("sample_kind_counts", sample_kind_counts)
print("user_feature_dim", USER_FEATURE_DIM)
print("preferredArea classes", list(feature_encoder.preferred_encoder.classes_))
print("model_known_poi_metadata", json.dumps(model_known_poi_metadata, ensure_ascii=False, indent=2))
assert train_samples


## 4. Class Imbalance Check, Dataset, Metrics, And GRU


In [ ]:
label_counts = pd.Series([sample["label_content_id"] for sample in train_samples], name="content_id").value_counts()
class_imbalance_summary = {
    "train_label_classes": int(label_counts.size),
    "train_label_rows": int(label_counts.sum()),
    "top1_count": int(label_counts.iloc[0]),
    "median_count": float(label_counts.median()),
    "singleton_classes": int((label_counts == 1).sum()),
    "top1_share": float(label_counts.iloc[0] / label_counts.sum()),
}
print(json.dumps(class_imbalance_summary, ensure_ascii=False, indent=2))
display(label_counts.head(20).rename("train_label_count").to_frame())

class TravelSequenceDataset(Dataset):
    def __init__(self, samples: list[dict[str, Any]]) -> None:
        self.samples = samples
    def __len__(self) -> int:
        return len(self.samples)
    def __getitem__(self, idx: int) -> dict[str, Any]:
        return self.samples[idx]

def collate_batch(batch: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
    user_features = torch.tensor(np.stack([item["user_features"] for item in batch]), dtype=torch.float32)
    lengths = torch.tensor([len(item["prefix"]) for item in batch], dtype=torch.long)
    sequences = torch.zeros((len(batch), int(lengths.max().item())), dtype=torch.long)
    for idx, item in enumerate(batch):
        sequences[idx, : len(item["prefix"])] = torch.tensor(item["prefix"], dtype=torch.long)
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    sample_kind_id = torch.tensor([0 if item["sample_kind"] == "first_poi" else 1 for item in batch], dtype=torch.long)
    return {"user_features": user_features, "sequences": sequences, "lengths": lengths, "labels": labels, "sample_kind_id": sample_kind_id}

def make_loader(samples: list[dict[str, Any]], batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(TravelSequenceDataset(samples), batch_size=batch_size, shuffle=shuffle, collate_fn=collate_batch)

def metric_at_k(labels: np.ndarray, scores: np.ndarray, ks: tuple[int, ...] = TOP_KS) -> dict[str, float]:
    order = np.argsort(-scores, axis=1)
    metrics = {}
    for k in ks:
        topk = order[:, : min(k, scores.shape[1])]
        hits = topk == labels[:, None]
        metrics[f"recall@{k}"] = float(hits.any(axis=1).mean())
        rr, ndcg = [], []
        for row_hits in hits:
            positions = np.flatnonzero(row_hits)
            rr.append(0.0 if len(positions) == 0 else 1.0 / float(positions[0] + 1))
            ndcg.append(0.0 if len(positions) == 0 else 1.0 / math.log2(float(positions[0] + 2)))
        metrics[f"mrr@{k}"] = float(np.mean(rr))
        metrics[f"ndcg@{k}"] = float(np.mean(ndcg))
    return metrics

def metric_by_sample_kind(labels: np.ndarray, scores: np.ndarray, sample_kind_ids: np.ndarray) -> dict[str, dict[str, float]]:
    metrics = {"overall": metric_at_k(labels, scores)}
    for kind_id, kind_name in [(0, "first_poi"), (1, "next_poi")]:
        mask = sample_kind_ids == kind_id
        metrics[kind_name] = metric_at_k(labels[mask], scores[mask]) if mask.any() else {}
    return metrics

def baseline_scores(train_items: list[dict[str, Any]], eval_items: list[dict[str, Any]], vocab_size: int) -> np.ndarray:
    counts = np.ones(vocab_size, dtype=np.float32) * 1e-6
    counts[list(SPECIAL_TOKEN_IDS)] = -np.inf
    for item in train_items:
        counts[int(item["label"])] += 1.0
    return np.repeat(counts[None, :], repeats=len(eval_items), axis=0)

@dataclass(frozen=True)
class GRUContentConfig:
    embedding_dim: int = 16
    hidden_dim: int = 64
    dropout: float = 0.2
    learning_rate: float = 1e-3
    batch_size: int = 64
    epochs: int = 8
    weight_decay: float = 1e-5

class GRUContentRecommender(nn.Module):
    def __init__(self, vocab_size: int, user_feature_dim: int, embedding_dim: int, hidden_dim: int, dropout: float) -> None:
        super().__init__()
        self.place_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.user_projection = nn.Sequential(nn.Linear(user_feature_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.gru = nn.GRU(input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True)
        self.output = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, vocab_size))
    def forward(self, user_features: torch.Tensor, sequences: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.place_embedding(sequences)
        gru_output, _ = self.gru(embedded)
        last_idx = torch.clamp(lengths - 1, min=0)
        gather_idx = last_idx.view(-1, 1, 1).expand(-1, 1, gru_output.size(2))
        sequence_context = torch.gather(gru_output, dim=1, index=gather_idx).squeeze(1)
        logits = self.output(torch.cat([sequence_context, self.user_projection(user_features)], dim=1))
        logits[:, list(SPECIAL_TOKEN_IDS)] = -1e9
        return logits

valid_labels = np.asarray([item["label"] for item in valid_samples], dtype=np.int64)
valid_sample_kind_ids = np.asarray([0 if item["sample_kind"] == "first_poi" else 1 for item in valid_samples], dtype=np.int64)
baseline_valid_scores = baseline_scores(train_samples, valid_samples, len(content_id_to_token))
baseline_valid = metric_by_sample_kind(valid_labels, baseline_valid_scores, valid_sample_kind_ids)
model = GRUContentRecommender(len(content_id_to_token), USER_FEATURE_DIM, 16, 64, 0.2).to(DEVICE)
batch = next(iter(make_loader(train_samples[:8], 4, shuffle=False)))
with torch.no_grad():
    logits_shape = model(batch["user_features"].to(DEVICE), batch["sequences"].to(DEVICE), batch["lengths"].to(DEVICE)).shape
assert logits_shape == (4, len(content_id_to_token))
baseline_valid


## 5. Common Candidate Constraints And API Simulators


In [ ]:
token_area_code = np.full(len(content_id_to_token), "", dtype=object)
valid_token_mask = np.zeros(len(content_id_to_token), dtype=bool)
content_id_to_area_code: dict[str, str] = {}
for row in model_poi_master.itertuples(index=False):
    token_id = int(row.token_id)
    token_area_code[token_id] = str(row.area_code)
    valid_token_mask[token_id] = True
    content_id_to_area_code[str(row.content_id)] = str(row.area_code)
valid_token_mask[list(SPECIAL_TOKEN_IDS)] = False
train_output_candidate_token_mask = np.zeros(len(content_id_to_token), dtype=bool)
for content_id in train_output_candidate_content_ids:
    train_output_candidate_token_mask[int(content_id_to_token[content_id])] = True
train_output_candidate_token_mask[list(SPECIAL_TOKEN_IDS)] = False

def area_token_mask(area_code: str) -> np.ndarray:
    normalized = normalize_area_code_5digit(area_code)
    return valid_token_mask & (token_area_code == normalized)

def apply_candidate_mask(
    logits: np.ndarray | torch.Tensor,
    area_code: str,
    excluded_content_ids: Iterable[Any] = (),
    restrict_to_train_outputs: bool = True,
) -> np.ndarray:
    scores = logits.detach().cpu().numpy().astype(np.float32).copy() if isinstance(logits, torch.Tensor) else np.asarray(logits, dtype=np.float32).copy()
    if scores.ndim != 1 or scores.shape[0] != len(content_id_to_token):
        raise ValueError("logits must be a 1D vector matching the model vocabulary size")
    candidate_mask = area_token_mask(area_code)
    if restrict_to_train_outputs:
        candidate_mask = candidate_mask & train_output_candidate_token_mask
    scores[~candidate_mask] = -np.inf
    scores[list(SPECIAL_TOKEN_IDS)] = -np.inf
    for content_id in excluded_content_ids:
        token_id = content_id_to_token.get(normalize_content_id_value(content_id))
        if token_id is not None:
            scores[int(token_id)] = -np.inf
    return scores

def top_content_ids_from_scores(scores: np.ndarray, top_k: int) -> list[dict[str, Any]]:
    finite = np.flatnonzero(np.isfinite(scores))
    order = finite[np.argsort(-scores[finite])]
    return [{"content_id": token_to_content_id[int(token_id)], "token_id": int(token_id), "score": float(scores[int(token_id)])} for token_id in order[:top_k]]

def fallback_top_k(area_code: str, excluded_content_ids: Iterable[Any] = (), top_k: int = 4) -> list[dict[str, Any]]:
    normalized_area_code = normalize_area_code_5digit(area_code)
    excluded = {normalize_content_id_value(value) for value in excluded_content_ids}
    rows = model_poi_master.loc[model_poi_master["area_code"].eq(normalized_area_code)]
    rows = rows.sort_values(["visit_count", "content_id"], ascending=[False, True], kind="mergesort")
    rows = rows.loc[~rows["content_id"].isin(excluded)].head(top_k)
    return [{"content_id": str(row.content_id), "token_id": int(row.token_id), "score": 1.0 - rank * 0.05} for rank, row in enumerate(rows.itertuples(index=False))]

def request_to_feature_frame(request: dict[str, Any]) -> pd.DataFrame:
    companion_value = request.get("companionCount", request.get("companion_count_bucket", MISSING_CONTEXT_TOKEN))
    persona_value = request.get("travelPersona", request.get("travel_persona", MISSING_CONTEXT_TOKEN))
    residence_value = request.get("residenceArea", request.get("residence_area_code", MISSING_CONTEXT_TOKEN))
    preferred_value = request.get("preferredArea", request.get("preferred_area_codes", [MISSING_CONTEXT_TOKEN]))
    row = {
        "companion_count_bucket": cap_companion_count(companion_value),
        "travel_persona": normalize_service_persona(persona_value),
        "residence_area_code": normalize_area_code_5digit(residence_value),
        "gender": normalize_context(request.get("gender", MISSING_CONTEXT_TOKEN)),
        "age_grp": normalize_context(request.get("ageGroup", request.get("age_grp", MISSING_CONTEXT_TOKEN))),
        "traveler_style": normalize_context(request.get("travelerStyle", request.get("traveler_style", MISSING_CONTEXT_TOKEN))),
        "preferred_area_codes": normalize_preferred_area_list(preferred_value),
    }
    return pd.DataFrame([row])

def encode_feature_request(request: dict[str, Any]) -> np.ndarray:
    return feature_encoder.transform(request_to_feature_frame(request))

def logits_for_prefix(model_obj: nn.Module, request: dict[str, Any], prefix_content_ids: list[str]) -> np.ndarray:
    prefix_tokens = tokens_for_model_prefix(prefix_content_ids)
    model_obj.eval()
    with torch.no_grad():
        logits = model_obj(
            torch.tensor(encode_feature_request(request), dtype=torch.float32, device=DEVICE),
            torch.tensor([prefix_tokens], dtype=torch.long, device=DEVICE),
            torch.tensor([len(prefix_tokens)], dtype=torch.long, device=DEVICE),
        )[0]
    return logits.detach().cpu().numpy()

def recommend_simulation(model_obj: nn.Module, request: dict[str, Any], top_k: int = 4) -> dict[str, Any]:
    area_code = normalize_area_code_5digit(request["areaCode"])
    original_sequence = [normalize_content_id_value(value) for value in request.get("contentIdSequence", [])]
    target_length = int(request["travelDuration"]) * COURSE_POIS_PER_DAY
    if not original_sequence:
        return {"mode": "fallback", "reason": "empty_sequence", "recommendations": fallback_top_k(area_code, [], top_k)}
    if len(original_sequence) >= target_length:
        return {"mode": "fallback", "reason": "full_course_sequence", "recommendations": fallback_top_k(area_code, original_sequence, top_k)}
    known_sequence = project_known_sequence(original_sequence)
    if not known_sequence:
        return {"mode": "fallback", "reason": "empty_known_sequence", "recommendations": fallback_top_k(area_code, original_sequence, top_k)}
    scores = apply_candidate_mask(logits_for_prefix(model_obj, request, known_sequence), area_code, original_sequence)
    recommendations = top_content_ids_from_scores(scores, top_k)
    if len(recommendations) < top_k:
        recommendations.extend(fallback_top_k(area_code, [*original_sequence, *[item["content_id"] for item in recommendations]], top_k - len(recommendations)))
    return {"mode": "gru", "known_sequence": known_sequence, "recommendations": recommendations[:top_k]}

def validate_required_content_ids(area_code: str, content_id_list: Iterable[Any], target_length: int) -> list[str]:
    normalized_area_code = normalize_area_code_5digit(area_code)
    required: set[str] = set()
    for value in content_id_list:
        content_id = normalize_content_id_value(value)
        if content_id not in content_id_to_token or content_id not in content_id_to_area_code:
            raise ValueError(f"contentIdList contains a POI outside the model candidate set: {content_id}")
        if content_id_to_area_code[content_id] != normalized_area_code:
            raise ValueError(f"contentIdList contains a POI outside areaCode {normalized_area_code}: {content_id}")
        required.add(content_id)
    if len(required) > target_length:
        raise ValueError("contentIdList cannot contain more unique items than target course length")
    return sorted(required)

def fallback_priority_for_content_id(content_id: str) -> tuple[int, str]:
    rows = model_poi_master.loc[model_poi_master["content_id"].eq(content_id)]
    if rows.empty:
        return (0, content_id)
    row = rows.iloc[0]
    return (int(row["visit_count"]), str(row["content_id"]))

def rank_required_candidates(required_ids: Iterable[str], scores: np.ndarray | None = None) -> list[str]:
    candidates = sorted({normalize_content_id_value(content_id) for content_id in required_ids})
    if scores is not None:
        scored_candidates = []
        for content_id in candidates:
            token_id = content_id_to_token.get(content_id)
            if token_id is not None and np.isfinite(scores[int(token_id)]):
                scored_candidates.append((float(scores[int(token_id)]), *fallback_priority_for_content_id(content_id), content_id))
        if scored_candidates:
            return [item[-1] for item in sorted(scored_candidates, key=lambda item: (-item[0], -item[1], item[2], item[3]))]
    return sorted(candidates, key=lambda content_id: (-fallback_priority_for_content_id(content_id)[0], fallback_priority_for_content_id(content_id)[1]))

def generate_course_simulation(model_obj: nn.Module, request: dict[str, Any]) -> dict[str, Any]:
    area_code = normalize_area_code_5digit(request["areaCode"])
    travel_duration = int(request["travelDuration"])
    target_length = travel_duration * COURSE_POIS_PER_DAY
    required = validate_required_content_ids(area_code, request.get("contentIdList", []), target_length)
    all_generated: list[str] = []
    daily_sequences: list[list[str]] = []
    gru_prefix_audit: list[dict[str, Any]] = []
    for day_index in range(1, travel_duration + 1):
        current_day_sequence: list[str] = []
        for _slot_index in range(COURSE_POIS_PER_DAY):
            step_request = request
            remaining_slots = target_length - len(all_generated)
            missing_required = sorted(set(required) - set(all_generated))
            known_day_prefix = project_known_sequence(current_day_sequence)
            gru_prefix_audit.append({"day_index": day_index, "slot_index": _slot_index + 1, "prefix_content_ids": known_day_prefix})
            scores = apply_candidate_mask(
                logits_for_prefix(model_obj, step_request, current_day_sequence),
                area_code,
                all_generated,
            )
            if missing_required and len(missing_required) >= remaining_slots:
                next_content_id = rank_required_candidates(missing_required, scores)[0]
                if not np.isfinite(scores[int(content_id_to_token[next_content_id])]):
                    next_content_id = rank_required_candidates(missing_required)[0]
            else:
                next_items = top_content_ids_from_scores(scores, 1) or fallback_top_k(area_code, all_generated, 1)
                if not next_items:
                    raise ValueError("No valid content_id candidates remain for course generation")
                next_content_id = next_items[0]["content_id"]
            all_generated.append(next_content_id)
            current_day_sequence.append(next_content_id)
        daily_sequences.append(current_day_sequence)
    return {"mode": "autoregressive_gru", "target_length": target_length, "required_content_ids": required, "content_id_sequence": all_generated, "daily_sequences": daily_sequences, "gru_prefix_audit": gru_prefix_audit}


## 6. Service-Condition Preflight Checks


In [ ]:
class DummyNextPoiModel(nn.Module):
    def __init__(self, vocab_size: int) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.call_count = 0
        self.call_prefix_tokens: list[list[int]] = []
        self.call_lengths: list[int] = []
    def forward(self, user_features: torch.Tensor, sequences: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        self.call_count += 1
        self.call_lengths.extend(int(length) for length in lengths.detach().cpu().tolist())
        for sequence, length in zip(sequences.detach().cpu().tolist(), lengths.detach().cpu().tolist()):
            self.call_prefix_tokens.append([int(token_id) for token_id in sequence[: int(length)]])
        return torch.arange(self.vocab_size, dtype=torch.float32, device=user_features.device).repeat(user_features.size(0), 1)

# 1. 5-digit area codes.
assert all(AREA_CODE_PATTERN.fullmatch(code) for code in AREA_CODE_TO_NAME)
assert seq["area_code"].map(lambda code: bool(AREA_CODE_PATTERN.fullmatch(str(code)))).all()
assert traveler_df["residence_area_code"].map(lambda code: code == MISSING_CONTEXT_TOKEN or bool(AREA_CODE_PATTERN.fullmatch(str(code)))).all()
assert all(code == MISSING_CONTEXT_TOKEN or AREA_CODE_PATTERN.fullmatch(code) for codes in traveler_df["preferred_area_codes"] for code in codes)

# 2. Raw TRAVEL_MISSION theme mapping follows the provided 7 service personas exactly.
expected_raw_theme_to_persona = {
    "22": 1,
    "1": 2,
    "4": 2,
    "3": 3,
    "6": 3,
    "2": 4,
    "9": 4,
    "11": 4,
    "25": 4,
    "5": 5,
    "8": 5,
    "27": 5,
    "24": 5,
    "21": 6,
    "10": 6,
    "23": 6,
    "26": 7,
    "12": 7,
    "13": 7,
}
assert RAW_THEME_TO_PERSONA == expected_raw_theme_to_persona
service_persona_ids = {str(i) for i in range(1, 8)}
mapped_personas = set(travel_df.loc[travel_df["travel_persona"].ne(MISSING_CONTEXT_TOKEN), "travel_persona"].unique())
assert mapped_personas.issubset(service_persona_ids)
assert normalize_service_persona(1) == "1" and normalize_service_persona(7) == "7"
assert {code for code in unmapped_theme_counts if code != MISSING_CONTEXT_TOKEN}.issubset({"7", "28"})

# 3. Companion counts are capped at 2.
assert set(travel_df["companion_count_bucket"].unique()).issubset({"0", "1", "2"})

# 4. preferredArea supports multi-hot input.
known_preferred_codes = [code for code in feature_encoder.preferred_encoder.classes_ if code != MISSING_CONTEXT_TOKEN][:2]
assert len(known_preferred_codes) == 2
multi_pref_request = request_to_feature_frame({"preferredArea": known_preferred_codes, "travelPersona": 3, "companionCount": 2})
assert multi_pref_request.iloc[0]["preferred_area_codes"] == known_preferred_codes
encoded_multi_pref = encode_feature_request({"preferredArea": known_preferred_codes, "travelPersona": 3, "companionCount": 2})
assert encoded_multi_pref.shape[1] == USER_FEATURE_DIM
assert int(encoded_multi_pref[0, -len(feature_encoder.preferred_encoder.classes_):].sum().item()) == 2

# 5. Split is by travel_id.
assert split_sets["train"].isdisjoint(split_sets["valid"])
assert split_sets["train"].isdisjoint(split_sets["test"])
assert split_sets["valid"].isdisjoint(split_sets["test"])

# 6. Stable full-data vocab is separate from train-known model POIs.
assert MIN_SEQUENCE_LEN == 1
assert content_id_to_token[PAD_TOKEN] == 0
assert content_id_to_token[BOS_TOKEN] == 1
assert min(token_id for content_id, token_id in content_id_to_token.items() if content_id not in {PAD_TOKEN, BOS_TOKEN}) >= 2
all_model_content_ids = {content_id for sequence in model_input["content_sequence"] for content_id in sequence}
assert all_model_content_ids.issubset(set(content_id_to_token))
assert {sample["label_content_id"] for sample in valid_samples}.issubset(set(content_id_to_token))
assert {sample["label_content_id"] for sample in test_samples}.issubset(set(content_id_to_token))
assert all_model_content_ids.issubset(master_content_ids)
assert train_known_input_content_ids.issubset(master_content_ids)
assert train_output_candidate_content_ids.issubset(master_content_ids)
single_poi_rows = model_input.loc[model_input["sequence_len"].eq(1)]
assert not single_poi_rows.empty
single_poi_content_id = normalize_content_id_value(single_poi_rows.iloc[0]["content_sequence"][0])
single_poi_samples = [sample for sample in train_samples + valid_samples + test_samples if sample["sample_kind"] == "first_poi" and sample["label_content_id"] == single_poi_content_id]
assert single_poi_samples and single_poi_samples[0]["prefix"] == [content_id_to_token[BOS_TOKEN]]
assert {"first_poi", "next_poi"}.issubset({sample["sample_kind"] for sample in train_samples})
assert {"first_poi", "next_poi"}.issubset({sample["sample_kind"] for sample in valid_samples})
assert {"first_poi", "next_poi"}.issubset({sample["sample_kind"] for sample in test_samples})
unseen_input_candidates = sorted(master_content_ids - train_known_input_content_ids)
assert unseen_input_candidates
print("model-known counts", json.dumps(model_known_poi_metadata, ensure_ascii=False, indent=2))

# 7. Samples are contained within one travel_id/day_index/area_code sequence.
assert all(str(sample["travel_day_id"]).count("::") == 2 for sample in train_samples + valid_samples + test_samples)
assert model_input.duplicated(["travel_id", "day_index", "area_code"]).sum() == 0
assert model_input.apply(lambda row: set(row["area_sequence"]) == {row["area_code"]}, axis=1).all()
assert cross_area_transition_count_before_area_split >= 0

# 8. Stable token artifact metadata exists in notebook state.
assert MODEL_VERSION and VOCAB_VERSION
assert poi_master["token_id"].notna().all()
assert poi_master["model_version"].eq(MODEL_VERSION).all()
assert poi_master["vocab_version"].eq(VOCAB_VERSION).all()

# 9. User ID-code features are categorical strings, not continuous numeric columns.
assert "day_index_context" not in SCALAR_CONTEXT_COLUMNS
assert "day_index_context" not in request_to_feature_frame({"travelPersona": 3, "companionCount": 1, "preferredArea": known_preferred_codes}).columns
for column in SCALAR_CONTEXT_COLUMNS:
    assert train_df[column].map(lambda value: isinstance(value, str) or pd.isna(value)).all()

# 10. Class imbalance was computed.
assert class_imbalance_summary["train_label_classes"] > 0
assert class_imbalance_summary["top1_share"] > 0

# 11. Missing context has explicit token semantics.
assert normalize_context(None) == MISSING_CONTEXT_TOKEN
assert normalize_area_code_5digit(None) == MISSING_CONTEXT_TOKEN
assert normalize_preferred_area_list(None) == [MISSING_CONTEXT_TOKEN]

sample_cid = str(model_poi_master.iloc[0]["content_id"])
assert token_to_content_id[int(content_id_to_token[sample_cid])] == sample_cid
area_code = None
for candidate_area_code in sorted(model_poi_master["area_code"].unique()):
    area_candidate_ids = model_poi_master.loc[model_poi_master["area_code"].eq(candidate_area_code), "content_id"].astype(str).tolist()
    area_known_input_ids = [content_id for content_id in area_candidate_ids if content_id in train_known_input_content_ids]
    area_output_ids = [content_id for content_id in area_candidate_ids if content_id in train_output_candidate_content_ids]
    area_non_output_ids = [content_id for content_id in area_candidate_ids if content_id not in train_output_candidate_content_ids]
    if (
        len(area_candidate_ids) >= COURSE_POIS_PER_DAY
        and len(area_known_input_ids) >= 2
        and len([content_id for content_id in area_output_ids if content_id not in area_known_input_ids[:2]]) >= 4
        and area_non_output_ids
    ):
        area_code = str(candidate_area_code)
        area_candidates = area_candidate_ids
        area_train_known_inputs = area_known_input_ids
        area_train_outputs = area_output_ids
        area_non_train_outputs = area_non_output_ids
        break
assert area_code is not None
assert len(area_candidates) >= COURSE_POIS_PER_DAY
assert not area_token_mask(area_code)[list(SPECIAL_TOKEN_IDS)].any()

known_a, known_b = area_train_known_inputs[:2]
unseen_vocab_content_id = next((content_id for content_id in area_candidates if content_id not in train_known_input_content_ids), unseen_input_candidates[0])
unknown = "999999999999"
assert unseen_vocab_content_id in content_id_to_token
assert unseen_vocab_content_id not in train_known_input_content_ids
assert project_known_sequence([unseen_vocab_content_id]) == []
assert project_known_sequence([known_a, unseen_vocab_content_id, known_b]) == [known_a, known_b]
masked_probe = apply_candidate_mask(np.arange(len(content_id_to_token), dtype=np.float32), area_code, [])
assert not np.isfinite(masked_probe[int(content_id_to_token[area_non_train_outputs[0]])])
fallback_probe = fallback_top_k(area_code, area_train_outputs, top_k=1)
assert fallback_probe and fallback_probe[0]["content_id"] in master_content_ids
assert fallback_probe[0]["content_id"] not in train_output_candidate_content_ids
base_request = {
    "areaCode": area_code,
    "travelDuration": "1",
    "travelPersona": 3,
    "companionCount": 3,
    "preferredArea": known_preferred_codes,
    "residenceArea": "11",
    "gender": "남",
    "ageGroup": "30",
    "travelerStyle": "4",
}
dummy = DummyNextPoiModel(len(content_id_to_token)).to(DEVICE)
bos_probe = DummyNextPoiModel(len(content_id_to_token)).to(DEVICE)
_ = logits_for_prefix(bos_probe, base_request, [])
assert bos_probe.call_prefix_tokens == [[content_id_to_token[BOS_TOKEN]]]
unknown_response = recommend_simulation(dummy, {**base_request, "contentIdSequence": [unknown]}, top_k=4)
assert unknown_response["mode"] == "fallback" and unknown_response["reason"] == "empty_known_sequence"
assert dummy.call_count == 0
unseen_vocab_response = recommend_simulation(dummy, {**base_request, "contentIdSequence": [unseen_vocab_content_id]}, top_k=4)
assert unseen_vocab_response["mode"] == "fallback" and unseen_vocab_response["reason"] == "empty_known_sequence"
assert dummy.call_count == 0
gru_response = recommend_simulation(dummy, {**base_request, "contentIdSequence": [known_a, unknown, known_b]}, top_k=4)
assert gru_response["mode"] == "gru" and dummy.call_count == 1
recommended_ids = [item["content_id"] for item in gru_response["recommendations"]]
assert known_a not in recommended_ids and known_b not in recommended_ids
assert all(content_id in train_output_candidate_content_ids for content_id in recommended_ids)
assert all(content_id_to_area_code[content_id] == area_code for content_id in recommended_ids)
course_response = generate_course_simulation(dummy, {**base_request, "contentIdList": area_train_known_inputs[:2]})
course_ids = course_response["content_id_sequence"]
assert len(course_ids) == COURSE_POIS_PER_DAY
assert len(course_ids) == len(set(course_ids))
assert set(area_train_known_inputs[:2]).issubset(course_ids)
assert course_response["required_content_ids"] == sorted(area_train_known_inputs[:2])
assert all(content_id_to_area_code[content_id] == area_code for content_id in course_ids)

# contentIdList is a required set, not an ordered prefix.
required_order_a = area_train_known_inputs[:3]
required_order_b = list(reversed(required_order_a))
set_dummy_a = DummyNextPoiModel(len(content_id_to_token)).to(DEVICE)
set_dummy_b = DummyNextPoiModel(len(content_id_to_token)).to(DEVICE)
set_response_a = generate_course_simulation(set_dummy_a, {**base_request, "contentIdList": required_order_a})
set_response_b = generate_course_simulation(set_dummy_b, {**base_request, "contentIdList": required_order_b})
assert set_response_a["required_content_ids"] == sorted(required_order_a)
assert set_response_b["required_content_ids"] == sorted(required_order_a)
assert set_response_a["content_id_sequence"] == set_response_b["content_id_sequence"]
assert set(required_order_a).issubset(set_response_a["content_id_sequence"])
forced_required = area_train_known_inputs[:COURSE_POIS_PER_DAY]
forced_dummy_a = DummyNextPoiModel(len(content_id_to_token)).to(DEVICE)
forced_dummy_b = DummyNextPoiModel(len(content_id_to_token)).to(DEVICE)
forced_response_a = generate_course_simulation(forced_dummy_a, {**base_request, "contentIdList": forced_required})
forced_response_b = generate_course_simulation(forced_dummy_b, {**base_request, "contentIdList": list(reversed(forced_required))})
assert forced_response_a["content_id_sequence"] == forced_response_b["content_id_sequence"]
assert set(forced_response_a["content_id_sequence"]) == set(forced_required)

# Multi-day generation must reset the GRU prefix at every day boundary while keeping global duplicate masking.
multi_day_dummy = DummyNextPoiModel(len(content_id_to_token)).to(DEVICE)
multi_day_required = area_train_known_inputs[:2]
multi_day_response = generate_course_simulation(multi_day_dummy, {**base_request, "travelDuration": "2", "contentIdList": multi_day_required})
multi_day_ids = multi_day_response["content_id_sequence"]
day1_ids = multi_day_ids[:COURSE_POIS_PER_DAY]
day2_ids = multi_day_ids[COURSE_POIS_PER_DAY:]
assert len(multi_day_ids) == COURSE_POIS_PER_DAY * 2
assert len(day1_ids) == COURSE_POIS_PER_DAY and len(day2_ids) == COURSE_POIS_PER_DAY
assert multi_day_response["daily_sequences"] == [day1_ids, day2_ids]
assert len(multi_day_ids) == len(set(multi_day_ids))
assert set(multi_day_required).issubset(multi_day_ids)
assert multi_day_dummy.call_count == COURSE_POIS_PER_DAY * 2
assert all(length <= COURSE_POIS_PER_DAY for length in multi_day_dummy.call_lengths)
assert multi_day_dummy.call_prefix_tokens[0] == [content_id_to_token[BOS_TOKEN]]
assert multi_day_dummy.call_prefix_tokens[COURSE_POIS_PER_DAY] == [content_id_to_token[BOS_TOKEN]]
day2_prefix_audit = [entry for entry in multi_day_response["gru_prefix_audit"] if entry["day_index"] == 2]
assert day2_prefix_audit
first_day2_prefix = day2_prefix_audit[0]["prefix_content_ids"]
assert first_day2_prefix == []
second_day2_prefix = day2_prefix_audit[1]["prefix_content_ids"]
assert second_day2_prefix
assert set(second_day2_prefix).issubset(set(day2_ids))
assert set(second_day2_prefix).isdisjoint(set(day1_ids))
assert all(set(entry["prefix_content_ids"]).issubset(set(multi_day_response["daily_sequences"][entry["day_index"] - 1])) for entry in multi_day_response["gru_prefix_audit"])
print("All 11 service-condition preflight checks passed.")
print("recommend sample", gru_response)
print("course sample", course_response)


## 7. Training Code Prepared But Not Executed

Set `RUN_FULL_TRAINING = True` only when ready to run the actual experiment. When enabled, the model checkpoint, full-data token vocabulary, POI Master, feature encoder, metrics, and version metadata are written under `artifacts/shared_next_poi_gru_experiment/{MODEL_VERSION}`.


In [ ]:
def run_epoch(model_obj: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer | None = None) -> float:
    is_train = optimizer is not None
    model_obj.train(is_train)
    loss_fn = nn.CrossEntropyLoss()
    total_loss, total_rows = 0.0, 0
    for batch in loader:
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        with torch.set_grad_enabled(is_train):
            logits = model_obj(batch["user_features"], batch["sequences"], batch["lengths"])
            loss = loss_fn(logits, batch["labels"])
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model_obj.parameters(), 5.0)
                optimizer.step()
        rows = int(batch["labels"].size(0))
        total_loss += float(loss.item()) * rows
        total_rows += rows
    return total_loss / max(total_rows, 1)

@torch.no_grad()
def predict_scores(model_obj: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    model_obj.eval()
    labels, scores, sample_kind_ids = [], [], []
    for batch in loader:
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        logits = model_obj(batch["user_features"], batch["sequences"], batch["lengths"])
        labels.append(batch["labels"].detach().cpu().numpy())
        scores.append(logits.detach().cpu().numpy())
        sample_kind_ids.append(batch["sample_kind_id"].detach().cpu().numpy())
    return np.concatenate(labels), np.concatenate(scores), np.concatenate(sample_kind_ids)

def train_one_config(config: GRUContentConfig) -> tuple[GRUContentRecommender, dict[str, Any]]:
    seed_everything(SEED)
    model_obj = GRUContentRecommender(len(content_id_to_token), USER_FEATURE_DIM, config.embedding_dim, config.hidden_dim, config.dropout).to(DEVICE)
    optimizer = torch.optim.AdamW(model_obj.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    train_loader = make_loader(train_samples, config.batch_size, shuffle=True)
    valid_loader = make_loader(valid_samples, config.batch_size, shuffle=False)
    history, best_state, best_recall = [], None, -1.0
    for epoch in range(1, config.epochs + 1):
        started = time.perf_counter()
        train_loss = run_epoch(model_obj, train_loader, optimizer)
        valid_loss = run_epoch(model_obj, valid_loader)
        labels, scores, sample_kind_ids = predict_scores(model_obj, valid_loader)
        valid_metrics = metric_at_k(labels, scores)
        valid_metrics_by_kind = metric_by_sample_kind(labels, scores, sample_kind_ids)
        row = {"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss, "epoch_seconds": time.perf_counter() - started, **valid_metrics}
        row["valid_metrics_by_kind"] = valid_metrics_by_kind
        history.append(row)
        print(json.dumps({**asdict(config), **row}, ensure_ascii=False))
        if valid_metrics["recall@10"] > best_recall:
            best_recall = valid_metrics["recall@10"]
            best_state = {key: value.detach().cpu().clone() for key, value in model_obj.state_dict().items()}
    if best_state is not None:
        model_obj.load_state_dict(best_state)
    return model_obj, {"config": asdict(config), "history": history, "best_valid_recall@10": best_recall}

EXPERIMENT_GRID = [
    GRUContentConfig(embedding_dim=16, hidden_dim=48, dropout=0.2, learning_rate=1e-3, batch_size=64, epochs=8),
    GRUContentConfig(embedding_dim=16, hidden_dim=64, dropout=0.2, learning_rate=1e-3, batch_size=64, epochs=8),
    GRUContentConfig(embedding_dim=32, hidden_dim=64, dropout=0.2, learning_rate=1e-3, batch_size=64, epochs=8),
]
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    experiment_results, best_model, best_result = [], None, None
    for config in EXPERIMENT_GRID:
        candidate_model, result = train_one_config(config)
        experiment_results.append(result)
        if best_result is None or result["best_valid_recall@10"] > best_result["best_valid_recall@10"]:
            best_model, best_result = candidate_model, result
    assert best_model is not None and best_result is not None
    test_loader = make_loader(test_samples, best_result["config"]["batch_size"], shuffle=False)
    test_labels, test_scores, test_sample_kind_ids = predict_scores(best_model, test_loader)
    test_metrics = metric_by_sample_kind(test_labels, test_scores, test_sample_kind_ids)
    checkpoint = {
        "model_state_dict": best_model.state_dict(),
        "model_class": "GRUContentRecommender",
        "model_version": MODEL_VERSION,
        "vocab_version": VOCAB_VERSION,
        "vocab_size": len(content_id_to_token),
        "user_feature_dim": USER_FEATURE_DIM,
        "config": best_result["config"],
        "max_sequence_len": MAX_SEQUENCE_LEN,
        "pad_token_id": content_id_to_token[PAD_TOKEN],
        "bos_token_id": content_id_to_token[BOS_TOKEN],
        "scalar_context_columns": SCALAR_CONTEXT_COLUMNS,
        "preferred_area_column": PREFERRED_AREA_COLUMN,
        "missing_context_token": MISSING_CONTEXT_TOKEN,
        "model_known_poi_metadata": model_known_poi_metadata,
    }
    torch.save(checkpoint, ARTIFACT_DIR / "best_shared_next_poi_gru.pt")
    with open(ARTIFACT_DIR / "content_id_vocab.json", "w", encoding="utf-8") as fp:
        json.dump({"model_version": MODEL_VERSION, "vocab_version": VOCAB_VERSION, "content_id_to_token": content_id_to_token, "token_to_content_id": token_to_content_id, "pad_token": PAD_TOKEN, "bos_token": BOS_TOKEN, "master_content_id_count": len(master_content_ids)}, fp, ensure_ascii=False, indent=2)
    with open(ARTIFACT_DIR / "model_known_pois.json", "w", encoding="utf-8") as fp:
        json.dump({**model_known_poi_metadata, "train_known_input_content_ids": sorted(train_known_input_content_ids), "train_output_candidate_content_ids": sorted(train_output_candidate_content_ids)}, fp, ensure_ascii=False, indent=2)
    poi_master.to_csv(ARTIFACT_DIR / "poi_master.csv", index=False, encoding="utf-8-sig")
    with open(ARTIFACT_DIR / "feature_encoder.pkl", "wb") as fp:
        pickle.dump(feature_encoder, fp)
    with open(ARTIFACT_DIR / "metrics.json", "w", encoding="utf-8") as fp:
        json.dump({"model_version": MODEL_VERSION, "vocab_version": VOCAB_VERSION, "coverage": coverage, "split_sizes": split_sizes, "sample_sizes": sample_sizes, "sample_kind_counts": sample_kind_counts, "model_known_poi_metadata": model_known_poi_metadata, "class_imbalance": class_imbalance_summary, "baseline_valid": baseline_valid, "test_metrics": test_metrics, "experiments": experiment_results, "best_config": best_result["config"]}, fp, ensure_ascii=False, indent=2)
    with open(ARTIFACT_DIR / "model_version.json", "w", encoding="utf-8") as fp:
        json.dump({"model_version": MODEL_VERSION, "vocab_version": VOCAB_VERSION, "created_by": "gru_shared_next_poi_masked_experiment.ipynb"}, fp, ensure_ascii=False, indent=2)
else:
    print("Training code is ready. Set RUN_FULL_TRAINING = True to start the full experiment.")
